In [36]:
import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, average_precision_score, f1_score
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split

from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from deepctr_torch.callbacks import EarlyStopping

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cpu'

In [20]:
games = pd.read_csv("Game Recommendations on Steam/games.csv")
recs = pd.read_csv("Game Recommendations on Steam/recommendations.csv")
desc = pd.read_csv("Steam Store Games/steam_description_data.csv")
media = pd.read_csv("Steam Store Games/steam_media_data.csv")
matched = pd.read_csv("games_steam_matched.csv")

In [22]:
desc = desc.rename(columns={"steam_appid": "appid"})
media = media.rename(columns={"steam_appid": "appid"})
matched = matched.merge(desc[['appid', 'about_the_game']], on='appid', how='left')
matched = matched.merge(media[['appid', 'header_image']], on='appid', how='left')
df_merged = matched 


In [23]:
df_merged.columns

Index(['app_id', 'title', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck', 'title_key', 'appid', 'name', 'release_date',
       'english', 'developer', 'publisher', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'achievements',
       'positive_ratings', 'negative_ratings', 'average_playtime',
       'median_playtime', 'owners', 'price', 'name_key', 'about_the_game_x',
       'header_image_x', 'game_title', 'about_the_game_y', 'header_image_y'],
      dtype='object')

In [24]:
df_merged["game_title"] = df_merged["title"].fillna(df_merged["name"])
df_merged = df_merged.drop(columns=["title", "name"], errors="ignore")
df_merged["header_image"] = df_merged["header_image_y"].fillna(df_merged["header_image_x"])
df_merged = df_merged.drop(columns=["header_image_x", "header_image_y"], errors="ignore")
df_merged["about_the_game"] = df_merged["about_the_game_y"].fillna(df_merged["about_the_game_x"])
df_merged = df_merged.drop(columns=["about_the_game_x", "about_the_game_y"], errors="ignore")

print(df_merged.columns)

Index(['app_id', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck', 'title_key', 'appid', 'release_date',
       'english', 'developer', 'publisher', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'achievements',
       'positive_ratings', 'negative_ratings', 'average_playtime',
       'median_playtime', 'owners', 'price', 'name_key', 'game_title',
       'header_image', 'about_the_game'],
      dtype='object')


In [25]:
df_merged.head()

,app_id,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,...,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,name_key,game_title,header_image,about_the_game
0,13500,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,...,657,101,307,312,200000-500000,8.59,prince of persia warrior withintm,Prince of Persia: Warrior Within™,https://steamcdn-a.akamaihd.net/steam/apps/135...,<p>Enter the dark underworld of Prince of Pers...
1,113020,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,...,7237,749,183,135,2000000-5000000,10.99,monaco what s yours is mine,Monaco: What's Yours Is Mine,https://steamcdn-a.akamaihd.net/steam/apps/113...,Monaco: What's Yours Is Mine is a single playe...
2,226560,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,...,549,383,0,0,50000-100000,10.99,escape dead island,Escape Dead Island,https://steamcdn-a.akamaihd.net/steam/apps/226...,"<img src=""https://steamcdn-a.akamaihd.net/stea..."
3,249050,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,...,7423,1046,648,295,500000-1000000,8.99,dungeon of the endlesstm,Dungeon of the ENDLESS™,https://steamcdn-a.akamaihd.net/steam/apps/249...,"<img src=""https://steamcdn-a.akamaihd.net/stea..."
4,250180,2015-09-14,True,False,False,Very Positive,90,5579,7.99,7.99,...,5229,553,212,242,500000-1000000,5.99,metal slug 3,METAL SLUG 3,https://steamcdn-a.akamaihd.net/steam/apps/250...,"<i>“METAL SLUG 3”</i>, the masterpiece in SNK’..."


In [26]:
df_final = recs.merge(
    df_merged,
    on="app_id",
    how="inner"
)

In [30]:
df_final.head()

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,date_release,win,...,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,name_key,game_title,header_image,about_the_game
0,304390,4,0,2017-02-17,False,11.5,2586,1,2017-02-13,True,...,34790,25920,951,696,2000000-5000000,12.49,for honortm,FOR HONOR™,https://steamcdn-a.akamaihd.net/steam/apps/304...,"Enter the chaos of war as a bold Knight, a bru..."
1,306130,0,0,2021-10-10,True,8.6,45425,5,2017-05-22,True,...,33719,11672,10659,3143,1000000-2000000,14.99,the elder scrolls online,The Elder Scrolls® Online,https://steamcdn-a.akamaihd.net/steam/apps/306...,Experience an ever-expanding story across all ...
2,238960,0,0,2017-11-25,True,538.8,88282,6,2013-10-23,True,...,71593,6117,5263,492,10000000-20000000,0.00,path of exile,Path of Exile,https://steamcdn-a.akamaihd.net/steam/apps/238...,"You are an Exile, struggling to survive on the..."
3,730,0,0,2021-11-30,False,157.5,63209,7,2012-08-21,True,...,2644404,402313,22494,6502,50000000-100000000,0.00,counter strike global offensive,Counter-Strike: Global Offensive,https://steamcdn-a.akamaihd.net/steam/apps/730...,Counter-Strike: Global Offensive (CS: GO) expa...
4,255710,0,0,2021-05-21,True,18.7,354512,8,2015-03-10,True,...,67553,6005,3225,444,5000000-10000000,22.99,cities skylines,Cities: Skylines,https://steamcdn-a.akamaihd.net/steam/apps/255...,<strong>Cities: Skylines</strong> is a modern ...


In [39]:
train_split = pd.read_csv("data/split/train_split.csv")
val_split   = pd.read_csv("data/split/val_split.csv")
test_split  = pd.read_csv("data/split/test_split.csv")
sampled_recs = pd.read_csv("data/sampled_recommendations.csv")


In [41]:
train_merged = train_split.merge(df_merged, on="app_id", how="left")
val_merged   = val_split.merge(df_merged, on="app_id", how="left")
test_merged  = test_split.merge(df_merged, on="app_id", how="left")
sampled_recs_merged  = sampled_recs.merge(df_merged, on="app_id", how="left")

In [42]:
sampled_recs_merged.head()

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,date_release,win,...,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,name_key,game_title,header_image,about_the_game
0,322330,0,0,2019-07-02,True,67.5,731,33484606,2016-04-21,True,...,108601.0,4206.0,1070.0,404.0,5000000-10000000,10.99,don t starve together,Don't Starve Together,https://steamcdn-a.akamaihd.net/steam/apps/322...,Don't Starve Together is the standalone multip...
1,433340,0,0,2020-01-24,True,32.3,731,26236770,2017-08-01,True,...,24205.0,965.0,1314.0,827.0,1000000-2000000,14.99,slime rancher,Slime Rancher,https://steamcdn-a.akamaihd.net/steam/apps/433...,"<img src=""https://steamcdn-a.akamaihd.net/stea..."
2,394360,2,0,2020-04-20,True,403.7,731,25992499,2016-06-06,True,...,34711.0,5251.0,9413.0,4181.0,1000000-2000000,34.99,hearts of iron iv,Hearts of Iron IV,https://steamcdn-a.akamaihd.net/steam/apps/394...,<strong>Victory is at your fingertips!</strong...
3,4700,0,0,2020-04-21,True,683.5,731,9845461,2007-11-27,True,...,10270.0,605.0,469.0,361.0,2000000-5000000,0.00,total war medieval ii definitive edition,Total War: MEDIEVAL II – Definitive Edition,https://steamcdn-a.akamaihd.net/steam/apps/470...,"<img src=""https://steamcdn-a.akamaihd.net/stea..."
4,255710,0,0,2020-05-02,True,37.6,731,33641355,2015-03-10,True,...,67553.0,6005.0,3225.0,444.0,5000000-10000000,22.99,cities skylines,Cities: Skylines,https://steamcdn-a.akamaihd.net/steam/apps/255...,<strong>Cities: Skylines</strong> is a modern ...


In [33]:
# Etiqueta a predecir
target = ['is_recommended']

# Features categóricas (strings / categorías)
sparse_features = [
    'rating', 'developer', 'publisher', 'platforms',
    'categories', 'genres', 'steamspy_tags'
]

# Features numéricas
dense_features = [
    'positive_ratio', 'user_reviews', 'price_final', 'price_original',
    'discount', 'positive_ratings', 'negative_ratings',
    'average_playtime', 'median_playtime', 'price',
    'required_age', 'achievements',
    'win', 'mac', 'linux', 'steam_deck',
    'helpful', 'funny', 'hours'
]


In [43]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

sampled_recs_merged[sparse_features] = sampled_recs_merged[sparse_features].fillna('-1')
sampled_recs_merged[dense_features] = sampled_recs_merged[dense_features].fillna(0)

# Label encoding para categóricas
for feat in sparse_features:
    lbe = LabelEncoder()
    sampled_recs_merged[feat] = lbe.fit_transform(sampled_recs_merged[feat].astype(str))

# Escalado [0,1] para numéricas
mms = MinMaxScaler(feature_range=(0, 1))
sampled_recs_merged[dense_features] = mms.fit_transform(sampled_recs_merged[dense_features])

In [44]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1.1. Texto base: descripción del juego (fallback al título)
sampled_recs_merged["text_input"] = (
    sampled_recs_merged["about_the_game"]
    .fillna(sampled_recs_merged["game_title"])
    .fillna("")
)

# 1.2. Cargar modelo de embeddings (es tipo BERT/derivado)
text_model = SentenceTransformer("all-MiniLM-L6-v2")  # dim = 384 aprox.

texts = sampled_recs_merged["text_input"].tolist()

text_emb = text_model.encode(
    texts,
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=True
)
# text_emb.shape -> (N, emb_dim)
emb_dim = text_emb.shape[1]
print("Dimensión embedding de texto:", emb_dim)


Batches: 100%|██████████| 1383/1383 [33:58<00:00,  1.47s/it]


Dimensión embedding de texto: 384


In [ ]:
# # Guardar
# # import numpy as np

# np.save("text_emb_sampled.npy", text_emb)

# # Cargar
# import numpy as np

# text_emb = np.load("text_emb_sampled.npy")
# print(text_emb.shape)

In [47]:
from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names

fixlen_feature_columns = (
    [SparseFeat(feat, vocabulary_size=df_final[feat].nunique(), embedding_dim=8)
     for feat in sparse_features]
    +
    [DenseFeat(feat, 1) for feat in dense_features]
    +
    [DenseFeat("text_emb", emb_dim)]  # 👈 nuestra feature de texto, multidimensional
)

dnn_feature_columns = fixlen_feature_columns
linear_feature_columns = fixlen_feature_columns

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


In [48]:
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
emb_dim = text_emb.shape[1]
print("Dim embedding:", emb_dim)

Dim embedding: 384


In [49]:
import numpy as np

def get_split_text_emb(split_df, id2idx, text_emb):
    idxs = [id2idx[rid] for rid in split_df["review_id"]]
    return text_emb[idxs]

train_text_emb = get_split_text_emb(train_merged, id2idx, text_emb)
val_text_emb   = get_split_text_emb(val_merged, id2idx, text_emb)
test_text_emb  = get_split_text_emb(test_merged, id2idx, text_emb)

print(train_text_emb.shape, val_text_emb.shape, test_text_emb.shape)


(68526, 384) (9906, 384) (9906, 384)


In [53]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Por si acaso
train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

for feat in sparse_features:
    lbe = LabelEncoder()
    
    # Fit con todas las categorías posibles (train+val+test)
    all_vals = pd.concat([
        train_merged[feat],
        val_merged[feat],
        test_merged[feat]
    ], axis=0).astype(str)
    
    lbe.fit(all_vals)
    
    # Transformar cada split
    train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
    val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
    test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))


In [54]:
import numpy as np

for feat in dense_features:
    train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
    val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
    test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

    train_merged[feat] = train_merged[feat].fillna(0)
    val_merged[feat]   = val_merged[feat].fillna(0)
    test_merged[feat]  = test_merged[feat].fillna(0)


In [55]:
# Sparse → int32
for feat in sparse_features:
    train_merged[feat] = train_merged[feat].astype('int32')
    val_merged[feat]   = val_merged[feat].astype('int32')
    test_merged[feat]  = test_merged[feat].astype('int32')

# Dense → float32
for feat in dense_features:
    train_merged[feat] = train_merged[feat].astype('float32')
    val_merged[feat]   = val_merged[feat].astype('float32')
    test_merged[feat]  = test_merged[feat].astype('float32')

# text_emb → float32
train_text_emb = train_text_emb.astype('float32')
val_text_emb   = val_text_emb.astype('float32')
test_text_emb  = test_text_emb.astype('float32')

# Target → float32
y_train = train_merged[target].values.astype('float32')
y_val   = val_merged[target].values.astype('float32')
y_test  = test_merged[target].values.astype('float32')


In [56]:
# TRAIN
train_model_input = {
    name: train_merged[name].values
    for name in feature_names
    if name != "text_emb"
}
train_model_input["text_emb"] = train_text_emb
y_train = train_merged[target].values

# VAL
val_model_input = {
    name: val_merged[name].values
    for name in feature_names
    if name != "text_emb"
}
val_model_input["text_emb"] = val_text_emb
y_val = val_merged[target].values

# TEST
test_model_input = {
    name: test_merged[name].values
    for name in feature_names
    if name != "text_emb"
}
test_model_input["text_emb"] = test_text_emb
y_test = test_merged[target].values


In [57]:
import torch
from deepctr_torch.models import DeepFM

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepFM(
    linear_feature_columns=linear_feature_columns,
    dnn_feature_columns=dnn_feature_columns,
    task="binary",
    device=device,
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["auc"],
)

model.fit(
    train_model_input,
    y_train,
    batch_size=256,
    epochs=10,
    verbose=2,
    validation_data=(val_model_input, y_val),
)

# Evaluar en test
pred_test = model.predict(test_model_input, batch_size=256)


cpu
Train on 68526 samples, validate on 9906 samples, 268 steps per epoch
Epoch 1/10
4s - loss:  9.5026 - auc:  0.5301 - val_auc:  0.5696
Epoch 2/10
3s - loss:  9.0753 - auc:  0.5338 - val_auc:  0.5735
Epoch 3/10
3s - loss:  9.0182 - auc:  0.5340 - val_auc:  0.5724
Epoch 4/10
3s - loss:  8.9517 - auc:  0.5365 - val_auc:  0.5732
Epoch 5/10
4s - loss:  8.8341 - auc:  0.5398 - val_auc:  0.5782
Epoch 6/10
4s - loss:  9.0283 - auc:  0.5351 - val_auc:  0.5774
Epoch 7/10
4s - loss:  9.0275 - auc:  0.5353 - val_auc:  0.5744
Epoch 8/10
4s - loss:  9.0275 - auc:  0.5347 - val_auc:  0.5748
Epoch 9/10
4s - loss:  9.0273 - auc:  0.5346 - val_auc:  0.5768
Epoch 10/10
4s - loss:  9.0270 - auc:  0.5347 - val_auc:  0.5736
